# RizeHire: BERT + ANN Model for ML Resume Classification
**Sandeep Kumar (2022UG1113) & Nikhil Sonkar (2022UG1111)**  
**IIIT Ranchi | Major Project | 2026**

### Pipeline Overview
This notebook trains an Artificial Neural Network (ANN) on top of BERT-derived features for classifying resumes as suitable (1) or unsuitable (0) for Machine Learning / Data Science roles.

**Feature Engineering Pipeline:**
1. **SBERT Embeddings** — Encode resume text + skills using Sentence-BERT (384-dim each)
2. **Cross-Encoder Scoring** — Compute semantic match between resume text and target ML job description
3. **Structured Features** — Years of experience, education level, portfolio availability
4. **Feature Fusion** — Concatenate all features into a rich vector
5. **Residual ANN** — Classify with skip connections, GELU activation, BatchNorm

**Dataset:** 4,500 resumes with text descriptions, skills, experience, and binary labels

---

## Step 1: Setup & install dependencies

In [ ]:
!pip install -q sentence-transformers torch scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.preprocessing import StandardScaler
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: {}'.format(device))
if device.type == 'cuda':
    print('GPU: {}'.format(torch.cuda.get_device_name(0)))

## Step 2: Load dataset

In [ ]:
from google.colab import files

print("Upload ml_resume_dataset_4500.csv:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Loaded {} rows, {} columns".format(len(df), len(df.columns)))
print("Columns: {}".format(list(df.columns)))
df.head()

In [ ]:
# Dataset structure:
# id, name, years_experience, highest_degree, skills, current_title, has_portfolio, raw_text, label
# label: 0 = not suitable for ML role, 1 = suitable for ML role

print("Label distribution:")
print(df['label'].value_counts())
print("\nLabel 0 (not suitable): {:.1f}%".format((df['label']==0).mean()*100))
print("Label 1 (suitable):     {:.1f}%".format((df['label']==1).mean()*100))

print("\nFeature summary:")
print("  years_experience: {:.1f} avg (range {}-{})".format(
    df['years_experience'].mean(), df['years_experience'].min(), df['years_experience'].max()))
print("  highest_degree:   {}".format(df['highest_degree'].value_counts().to_dict()))
print("  has_portfolio:    {}".format(df['has_portfolio'].value_counts().to_dict()))
print("  current_title:    {} unique titles".format(df['current_title'].nunique()))
print("  raw_text:         {:.0f} avg chars".format(df['raw_text'].str.len().mean()))
print("  skills:           {:.0f} avg chars".format(df['skills'].str.len().mean()))

# Show the difference between label 0 and label 1
print("\nLabel 0 vs Label 1:")
for label_val in [0, 1]:
    sub = df[df['label']==label_val]
    print("  Label {}: avg_exp={:.1f}yr, portfolio={:.0f}%, avg_skills={:.1f}".format(
        label_val, sub['years_experience'].mean(),
        sub['has_portfolio'].mean()*100,
        sub['skills'].str.count(',').mean()+1))

## Step 3: Feature engineering

In [ ]:
# Prepare text columns
resume_texts = df['raw_text'].astype(str).tolist()
skills_texts = df['skills'].astype(str).tolist()
title_texts = df['current_title'].astype(str).tolist()

# Create a combined resume text: raw_text + skills + title
combined_texts = []
for i in range(len(df)):
    combined = "{} Skills: {}. Current role: {}.".format(
        resume_texts[i], skills_texts[i], title_texts[i])
    combined_texts.append(combined)

print("Sample combined text:")
print("  [Label 1]: {}".format(combined_texts[df[df['label']==1].index[0]]))
print("  [Label 0]: {}".format(combined_texts[df[df['label']==0].index[0]]))

# Encode structured features
degree_map = {'Bachelors': 0, 'Masters': 1, 'PhD': 2}
structured_features = np.column_stack([
    df['years_experience'].values,
    df['highest_degree'].map(degree_map).values,
    df['has_portfolio'].astype(int).values,
    df['skills'].str.count(',').values + 1,  # number of skills
    df['raw_text'].str.len().values / 100.0,  # normalized text length
    df['skills'].str.len().values / 100.0,    # normalized skills length
]).astype(np.float32)

print("\nStructured features shape: {}".format(structured_features.shape))
print("Features: [years_exp, degree_level, has_portfolio, skill_count, text_len, skills_len]")

## Step 4: Generate BERT embeddings (SBERT + Cross-Encoder)

In [ ]:
# Load SBERT for semantic embeddings
print("Loading Sentence-BERT (all-MiniLM-L6-v2)...")
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("SBERT loaded! Embedding dim: {}".format(sbert.get_sentence_embedding_dimension()))

# Load Cross-Encoder for resume-job matching
print("\nLoading Cross-Encoder (ms-marco-MiniLM-L-6-v2)...")
cross_enc = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-Encoder loaded!")

In [ ]:
# Generate SBERT embeddings for combined resume text
print("Encoding combined resume texts (raw_text + skills + title)...")
resume_embeddings = sbert.encode(combined_texts, batch_size=128,
                                 show_progress_bar=True, convert_to_numpy=True)
print("Resume embeddings: {}".format(resume_embeddings.shape))

# Generate SBERT embeddings for skills separately
print("\nEncoding skills texts...")
skills_embeddings = sbert.encode(skills_texts, batch_size=128,
                                 show_progress_bar=True, convert_to_numpy=True)
print("Skills embeddings: {}".format(skills_embeddings.shape))

In [ ]:
# Cross-Encoder: Score resume against a target ML job description
# This simulates the real-world matching in RizeHire

target_job_desc = (
    "We are looking for a Machine Learning Engineer with strong experience in "
    "Python, deep learning frameworks (TensorFlow, PyTorch), model deployment, "
    "MLOps, data pipelines, and statistical analysis. The ideal candidate has "
    "experience with NLP, computer vision, or time-series forecasting, and can "
    "deploy production ML systems using Docker, Kubernetes, and cloud platforms."
)

print("Target job description:")
print("  {}".format(target_job_desc[:120]))

# Generate cross-encoder scores
print("\nComputing Cross-Encoder scores (resume <-> ML job)...")
pairs = [[combined_texts[i], target_job_desc] for i in range(len(combined_texts))]
cross_scores = cross_enc.predict(pairs, batch_size=128, show_progress_bar=True)
cross_scores = np.array(cross_scores, dtype=np.float32).reshape(-1, 1)

print("Cross-encoder scores shape: {}".format(cross_scores.shape))
print("Score range: [{:.3f}, {:.3f}]".format(cross_scores.min(), cross_scores.max()))

# Also get SBERT embedding of the job description for cosine similarity
job_embedding = sbert.encode([target_job_desc], convert_to_numpy=True)

# Cosine similarity between each resume and the job description
from numpy.linalg import norm
cosine_sims = np.sum(resume_embeddings * job_embedding, axis=1) / (
    norm(resume_embeddings, axis=1) * norm(job_embedding, axis=1) + 1e-8)
cosine_sims = cosine_sims.reshape(-1, 1).astype(np.float32)

print("\nCosine similarity shape: {}".format(cosine_sims.shape))
print("Cosine sim range: [{:.3f}, {:.3f}]".format(cosine_sims.min(), cosine_sims.max()))

# Check: do scores differ between label 0 and 1?
labels = df['label'].values
print("\nCross-encoder scores by label:")
print("  Label 0 (not suitable): mean={:.3f}".format(cross_scores[labels==0].mean()))
print("  Label 1 (suitable):     mean={:.3f}".format(cross_scores[labels==1].mean()))
print("Cosine similarity by label:")
print("  Label 0: mean={:.3f}".format(cosine_sims[labels==0].mean()))
print("  Label 1: mean={:.3f}".format(cosine_sims[labels==1].mean()))

In [ ]:
# Interaction features between resume and job embeddings
print("Computing interaction features...")

# Element-wise product (resume * job) — captures matching dimensions
job_emb_broadcast = np.repeat(job_embedding, len(resume_embeddings), axis=0)
elem_product = resume_embeddings * job_emb_broadcast  # 384-dim

# Element-wise absolute difference — captures mismatches
elem_diff = np.abs(resume_embeddings - job_emb_broadcast)  # 384-dim

print("Element-wise product: {}".format(elem_product.shape))
print("Element-wise diff: {}".format(elem_diff.shape))

In [ ]:
# COMBINE ALL FEATURES
combined_features = np.concatenate([
    resume_embeddings,     # 384-dim: SBERT embedding of resume+skills+title
    skills_embeddings,     # 384-dim: SBERT embedding of skills
    elem_product,          # 384-dim: resume*job element-wise
    elem_diff,             # 384-dim: |resume-job| element-wise
    cross_scores,          # 1-dim: cross-encoder similarity score
    cosine_sims,           # 1-dim: cosine similarity resume<->job
    structured_features,   # 6-dim: years_exp, degree, portfolio, skill_count, etc.
], axis=1)

print("=" * 60)
print("FINAL FEATURE VECTOR: {}".format(combined_features.shape))
print("=" * 60)
print("  SBERT resume embedding:      384 dims")
print("  SBERT skills embedding:      384 dims")
print("  Element-wise product:        384 dims")
print("  Element-wise |diff|:         384 dims")
print("  Cross-encoder score:           1 dim")
print("  Cosine similarity:             1 dim")
print("  Structured features:           6 dims")
print("  " + "-" * 40)
print("  TOTAL:                      {} dims".format(combined_features.shape[1]))

## Step 5: Normalize & split data

In [ ]:
# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(combined_features)

labels = df['label'].values
num_classes = len(np.unique(labels))

# Split: 80% train, 10% val, 10% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, labels, test_size=0.1, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.111, random_state=42, stratify=y_train_val)

print("Training:   {} samples".format(len(X_train)))
print("Validation: {} samples".format(len(X_val)))
print("Test:       {} samples".format(len(X_test)))

# Class weights for imbalanced data (70/30 split)
class_counts = Counter(y_train)
total = len(y_train)
class_weights = {c: total / (num_classes * n) for c, n in class_counts.items()}
weight_tensor = torch.FloatTensor([class_weights[i] for i in range(num_classes)]).to(device)
print("\nClass distribution: {}".format(dict(class_counts)))
print("Class weights: {}".format(class_weights))

# Convert to tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

# Weighted sampler
sample_weights = [class_weights[y] for y in y_train]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, sampler=sampler)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)

print("\nDataLoaders ready (batch_size=64, weighted sampling)")

## Step 6: Define Residual ANN architecture

**Architecture:** Input -> 512 -> ResBlock -> ResBlock -> 256 -> 128 -> 2  
With GELU activation, BatchNorm, skip connections, and graduated dropout

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.activation(x + self.block(x)))


class ResumeClassifierANN(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        self.res1 = ResidualBlock(512, dropout=0.3)
        self.res2 = ResidualBlock(512, dropout=0.25)
        self.reduce = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.res1(x)
        x = self.res2(x)
        x = self.reduce(x)
        return self.classifier(x)


input_dim = X_scaled.shape[1]
model = ResumeClassifierANN(input_dim=input_dim, num_classes=num_classes).to(device)

print("Residual ANN Architecture:")
print("Input: {} -> 512 -> ResBlock x2 -> 256 -> 128 -> {}".format(input_dim, num_classes))
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("\nTotal parameters: {:,}".format(total_params))

## Step 7: Train the model

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-3)

EPOCHS = 80
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

best_val_acc = 0
best_val_f1 = 0
patience = 20
patience_counter = 0

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

print("Training BERT+ANN for {} epochs...".format(EPOCHS))
print("Optimizer: AdamW (lr=0.0005, wd=1e-3)")
print("Loss: CrossEntropy + class weights + label smoothing (0.05)")
print("=" * 80)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == batch_y).sum().item()

    scheduler.step(epoch)
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_labels, all_preds, average='weighted')

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        marker = ' *** BEST (Acc:{:.2f}% F1:{:.2f}%)'.format(val_acc*100, val_f1*100)
    else:
        patience_counter += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0 or marker:
        print("Epoch [{:3d}/{}] Train:{:.2f}% Val:{:.2f}% Loss:{:.4f}/{:.4f} LR:{:.6f}{}".format(
            epoch+1, EPOCHS, train_acc*100, val_acc*100, train_loss, val_loss, current_lr, marker))

    if patience_counter >= patience:
        print("\nEarly stopping at epoch {}".format(epoch+1))
        break

print("\n" + "=" * 80)
print("Training complete! Best val acc: {:.2f}%, Best val F1: {:.2f}%".format(
    best_val_acc*100, best_val_f1*100))

## Step 8: Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_acc']) + 1)

axes[0].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-', lw=2, label='Training', alpha=0.8)
axes[0].plot(epochs_range, [a*100 for a in history['val_acc']], 'g-', lw=2, label='Validation', alpha=0.8)
axes[0].axhline(y=best_val_acc*100, color='r', ls='--', alpha=0.7, label='Best: {:.2f}%'.format(best_val_acc*100))
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('BERT + Residual ANN Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_loss'], 'b-', lw=2, label='Training', alpha=0.8)
axes[1].plot(epochs_range, history['val_loss'], 'r-', lw=2, label='Validation', alpha=0.8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['lr'], 'purple', lw=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Cosine Annealing LR', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bert_ann_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_training_curves.png")

## Step 9: Evaluate on test set

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()

with torch.no_grad():
    test_out = model(X_test_t)
    _, y_pred = torch.max(test_out, 1)
    probs = torch.softmax(test_out, dim=1).cpu().numpy()

y_pred_np = y_pred.cpu().numpy()
y_test_np = y_test_t.cpu().numpy()

test_acc = accuracy_score(y_test_np, y_pred_np)
test_prec = precision_score(y_test_np, y_pred_np, average='weighted')
test_rec = recall_score(y_test_np, y_pred_np, average='weighted')
test_f1 = f1_score(y_test_np, y_pred_np, average='weighted')

print("=" * 60)
print("   BERT + RESIDUAL ANN — TEST RESULTS")
print("=" * 60)
print("   Accuracy:  {:.2f}%".format(test_acc * 100))
print("   Precision: {:.2f}%".format(test_prec * 100))
print("   Recall:    {:.2f}%".format(test_rec * 100))
print("   F1-Score:  {:.2f}%".format(test_f1 * 100))
print("=" * 60)

print("\nClassification Report:")
print(classification_report(y_test_np, y_pred_np, target_names=['Not Suitable', 'Suitable']))

## Step 10: Confusion matrix & ROC curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test_np, y_pred_np)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Not Suitable', 'Suitable'],
            yticklabels=['Not Suitable', 'Suitable'])
ax1.set_xlabel('Predicted', fontweight='bold')
ax1.set_ylabel('Actual', fontweight='bold')
ax1.set_title('Confusion Matrix (Acc: {:.2f}%)'.format(test_acc*100), fontweight='bold')

fpr, tpr, _ = roc_curve(y_test_np, probs[:, 1])
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, 'b-', lw=2, label='BERT+ANN (AUC = {:.3f})'.format(roc_auc))
ax2.plot([0, 1], [0, 1], 'r--', lw=1, label='Random')
ax2.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bert_ann_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_evaluation.png")

## Step 11: Pipeline flow diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')

c_in = '#3498DB'; c_bert = '#E74C3C'; c_feat = '#2ECC71'; c_ann = '#9B59B6'; c_out = '#F39C12'

def box(ax, x, y, w, h, text, color, fs=9):
    ax.add_patch(plt.Rectangle((x,y), w, h, facecolor=color, edgecolor='white', alpha=0.85, lw=2, zorder=2))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fs, fontweight='bold', color='white', zorder=3, multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1), arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

ax.text(8, 9.5, 'RizeHire: BERT + Residual ANN Classification Pipeline', ha='center', fontsize=16, fontweight='bold', color='#2C3E50')

# Row 1: Inputs
box(ax, 0.5, 7.8, 2.8, 0.8, 'Resume Text\n(raw_text)', c_in)
box(ax, 3.8, 7.8, 2.8, 0.8, 'Skills List\n(skills)', c_in)
box(ax, 7.1, 7.8, 2.8, 0.8, 'Job Description\n(ML Role Target)', c_in)
box(ax, 10.4, 7.8, 3.0, 0.8, 'Structured Features\n(exp, degree, portfolio)', c_in, fs=8)

for x in [1.9, 5.2, 8.5, 11.9]: arrow(ax, x, 7.8, x, 7.0)

# Row 2: BERT
box(ax, 0.5, 6.2, 2.8, 0.8, 'SBERT Encoder\n384-dim embedding', c_bert)
box(ax, 3.8, 6.2, 2.8, 0.8, 'SBERT Encoder\n384-dim embedding', c_bert)
box(ax, 7.1, 6.2, 2.8, 0.8, 'Cross-Encoder\nSimilarity Score', c_bert)
box(ax, 10.4, 6.2, 3.0, 0.8, 'Feature Encoding\n6-dim vector', c_bert)

for xs, xe in [(1.9, 3.5), (5.2, 5.2), (8.5, 8.0), (11.9, 11.0)]: arrow(ax, xs, 6.2, xe, 5.5)

# Row 3: Feature engineering
box(ax, 1.0, 4.7, 3.5, 0.8, 'Element-wise Product\nResume * Job (384-dim)', c_feat, fs=8)
box(ax, 5.0, 4.7, 3.5, 0.8, '|Resume - Job|\nAbsolute Diff (384-dim)', c_feat, fs=8)
box(ax, 9.0, 4.7, 3.0, 0.8, 'Cosine Similarity\n+ Cross-Encoder', c_feat, fs=8)

for x in [2.75, 6.75, 10.5]: arrow(ax, x, 4.7, 7.75, 4.0)

# Row 4: Fusion
box(ax, 4.5, 3.2, 6.5, 0.8, 'Feature Fusion ({}-dim) -> StandardScaler'.format(input_dim), c_feat)
arrow(ax, 7.75, 3.2, 7.75, 2.6)

# Row 5: ANN
box(ax, 3.0, 1.8, 9.5, 0.8, 'Residual ANN: {} -> 512 -> ResBlock x2 -> 256 -> 128 -> Softmax\n(GELU + BatchNorm + Dropout + Skip Connections)'.format(input_dim), c_ann, fs=9)
arrow(ax, 7.75, 1.8, 7.75, 1.2)

# Row 6: Output
box(ax, 5.0, 0.4, 5.5, 0.8, 'Classification: Suitable / Not Suitable\nAccuracy: {:.2f}% | F1: {:.2f}%'.format(test_acc*100, test_f1*100), c_out)

plt.tight_layout()
plt.savefig('pipeline_flow.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: pipeline_flow.png")

## Step 12: Research paper comparison

In [ ]:
studies = [
    ('Sabarirajan et al. (2025)', 'BERT+RoBERTa', 87, True, False),
    ('Deshmukh & Raut (2024)', 'NLP+Hybrid ML', 88, False, False),
    ('Wankhede et al. (2024)', 'NLP+Decision Tree', 96, False, False),
    ('More & Kene (2024)', 'TF-IDF+SVM/RF', 90, False, False),
    ('Jyothsna et al. (2024)', 'Cosine+AdaBoost', 87, False, False),
    ('Bevara et al. (2025)', 'SBERT Embeddings', 78, False, False),
    ('Sriranjani (2025)', 'BERT+Random Forest', 87, False, False),
    ('Wilson & Caliskan (2025)', 'GPT-4 Zero-Shot', 82, False, False),
    ('Lo (2025)', 'BERT Fine-Tuning', 85, False, False),
    ('Tejaswini et al. (2022)', 'TF-IDF+ML', 85, False, False),
    ('RizeHire (Ours)', 'CrossEnc+SBERT+ResANN', round(test_acc*100, 2), True, True),
]

fig, ax = plt.subplots(figsize=(14, 7))
names = [s[0] for s in studies]
accs = [s[2] for s in studies]
colors = ['#27AE60' if 'Ours' in s[0] else '#95A5A6' for s in studies]

bars = ax.barh(names, accs, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Accuracy / F1-Score (%)', fontweight='bold')
ax.set_title('Performance Comparison with Latest Research (2022-2025)', fontweight='bold', fontsize=14)
ax.set_xlim(0, 110)
ax.grid(True, alpha=0.3, axis='x')

for bar, s in zip(bars, studies):
    label = '{}%'.format(s[2])
    extras = []
    if s[3]: extras.append('XAI')
    if s[4]: extras.append('Real-Time')
    if extras: label += ' + ' + ' + '.join(extras)
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, label,
            va='center', fontweight='bold', fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#27AE60', label='RizeHire (Ours)'),
    Patch(facecolor='#95A5A6', label='Other Studies'),
], loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('bert_ann_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_comparison.png")

## Step 13: SHAP explainability

In [ ]:
!pip install -q shap

In [ ]:
import shap

def predict_fn(X):
    model.eval()
    with torch.no_grad():
        t = torch.FloatTensor(X).to(device)
        return torch.softmax(model(t), dim=1).cpu().numpy()

feature_names = (
    ['Resume_SBERT_{}'.format(i) for i in range(384)] +
    ['Skills_SBERT_{}'.format(i) for i in range(384)] +
    ['Resume*Job_{}'.format(i) for i in range(384)] +
    ['|Res-Job|_{}'.format(i) for i in range(384)] +
    ['CrossEncoder_Score', 'Cosine_Similarity',
     'Years_Experience', 'Degree_Level', 'Has_Portfolio',
     'Skill_Count', 'Text_Length', 'Skills_Length']
)

background = X_train[:50]
explainer = shap.KernelExplainer(predict_fn, background)

test_samples = X_test[:15]
print("Computing SHAP values (this takes a few minutes)...")
shap_values = explainer.shap_values(test_samples)
print("Done!")

# Focus on interpretable features (last 8)
interp_idx = list(range(len(feature_names)-8, len(feature_names)))
interp_names = [feature_names[i] for i in interp_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(ax1)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
shap.summary_plot(sv, test_samples, feature_names=feature_names, max_display=20, show=False)
ax1.set_title('SHAP: Top 20 features', fontweight='bold')

plt.sca(ax2)
shap.summary_plot(sv[:, interp_idx], test_samples[:, interp_idx],
                  feature_names=interp_names, max_display=8, show=False)
ax2.set_title('SHAP: Interpretable features', fontweight='bold')

plt.tight_layout()
plt.savefig('bert_ann_shap.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_shap.png")

## Step 14: Feature group importance

In [ ]:
sv_abs = np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values)

groups = {
    'SBERT Resume Embedding': sv_abs[:, :384].mean(),
    'SBERT Skills Embedding': sv_abs[:, 384:768].mean(),
    'Resume*Job Product': sv_abs[:, 768:1152].mean(),
    '|Resume-Job| Diff': sv_abs[:, 1152:1536].mean(),
    'Cross-Encoder + Cosine': sv_abs[:, 1536:1538].mean(),
    'Structured Features': sv_abs[:, 1538:].mean(),
}

fig, ax = plt.subplots(figsize=(10, 5))
g_names = list(groups.keys())
g_vals = list(groups.values())
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6', '#F39C12', '#1ABC9C']

bars = ax.barh(g_names, g_vals, color=colors, edgecolor='white', height=0.5)
ax.set_xlabel('Mean |SHAP Value|', fontweight='bold')
ax.set_title('Feature Group Importance in BERT+ANN', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, g_vals):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            '{:.4f}'.format(val), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_group_importance.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: feature_group_importance.png")

## Step 15: Save model & download results

In [ ]:
save_dict = {
    'model_state_dict': model.state_dict(),
    'input_dim': input_dim,
    'num_classes': num_classes,
    'architecture': '{}->512->ResBlockx2->256->128->{}'.format(input_dim, num_classes),
    'test_accuracy': test_acc,
    'test_precision': test_prec,
    'test_recall': test_rec,
    'test_f1': test_f1,
    'best_val_accuracy': best_val_acc,
    'best_val_f1': best_val_f1,
    'training_history': history,
}
torch.save(save_dict, 'rizehire_bert_ann_final.pth')
print("Model saved!")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print("\n" + "=" * 70)
print("       FINAL RESULTS — RizeHire BERT+ANN")
print("=" * 70)
print("  BERT Models:      SBERT + Cross-Encoder (MiniLM-L6)")
print("  ANN:              {}->512->ResBlockx2->256->128->{}".format(input_dim, num_classes))
print("  Dataset:          4,500 resumes (ML role classification)")
print("  Train/Val/Test:   {}/{}/{}".format(len(X_train), len(X_val), len(X_test)))
print("  GPU:              {}".format(gpu_name))
print("  " + "-" * 45)
print("  Test Accuracy:    {:.2f}%".format(test_acc * 100))
print("  Test Precision:   {:.2f}%".format(test_prec * 100))
print("  Test Recall:      {:.2f}%".format(test_rec * 100))
print("  Test F1-Score:    {:.2f}%".format(test_f1 * 100))
print("  Best Val Acc:     {:.2f}%".format(best_val_acc * 100))
print("  AUC-ROC:          {:.3f}".format(roc_auc))
print("=" * 70)

In [ ]:
from google.colab import files

for f in ['rizehire_bert_ann_final.pth',
          'bert_ann_training_curves.png',
          'bert_ann_evaluation.png',
          'bert_ann_comparison.png',
          'bert_ann_shap.png',
          'pipeline_flow.png',
          'feature_group_importance.png']:
    try:
        files.download(f)
        print('Downloaded: {}'.format(f))
    except Exception:
        print('Download manually: {}'.format(f))

print("\nDone! Share the PNG files with your professor.")